In [38]:
import numpy as np
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [39]:
df.dropna(inplace=True)


In [40]:
df.drop_duplicates(inplace=True)

In [41]:
df=df[~(df['clean_comment'].str.strip()=='')]

In [42]:
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [43]:
# %pip uninstall nltk -

In [44]:
import nltk
print(nltk.__file__)

print(dir(nltk)[:20])   # show some attributes

C:\Users\ADMIN\AppData\Roaming\Python\Python313\site-packages\nltk\__init__.py
['ARLSTem', 'ARLSTem2', 'AbstractLazySequence', 'AffixTagger', 'AlignedSent', 'Alignment', 'AnnotationTask', 'ApplicationExpression', 'Assignment', 'BigramAssocMeasures', 'BigramCollocationFinder', 'BigramTagger', 'BinaryMaxentFeatureEncoding', 'BlanklineTokenizer', 'BllipParser', 'BottomUpChartParser', 'BottomUpLeftCornerChartParser', 'BottomUpProbabilisticChartParser', 'Boxer', 'BrillTagger']


In [45]:
# %pip install --no-cache-dir nltk==3.8.1

In [46]:
import nltk
print(nltk.__file__)
print(nltk.__version__)
nltk.download('stopwords')

C:\Users\ADMIN\AppData\Roaming\Python\Python313\site-packages\nltk\__init__.py
3.8.1


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [47]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [48]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [49]:
def preprocess_comment(comment):
    comment=comment.lower()
    
    # removing whitespaces
    comment=comment.strip()
    
    # removing newline characters
    comment=re.sub(r'\n',' ',comment)
        # remove non alphanumeric characters
    comment=re.sub(r'[^A-Za-z0-9\s!?.,]','',comment)
    
    # remove stopwords but retain important ones for sentiment analysis
    stop_words=set(stopwords.words('english'))-{'not','but','however','no','yet'}
    comment=' '.join([word for word in comment.split() if word not in stop_words])
    
    lemmatizer=WordNetLemmatizer()
    comment=' '.join([lemmatizer.lemmatize(word) for word in comment.split()])
    
    return comment
    

In [50]:
df['clean_comment']=df['clean_comment'].apply(preprocess_comment)

In [51]:
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [52]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [53]:
vectorizer=CountVectorizer(max_features=5000)

In [54]:
X=df['clean_comment']
y=df['category']

In [55]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(36793, 5000))

In [56]:
X.shape

(36793, 5000)

In [57]:
y

0        1
1        1
2       -1
3        0
4        1
        ..
37244    0
37245    1
37246    0
37247    1
37248    0
Name: category, Length: 36793, dtype: int64

In [60]:
import dagshub
import mlflow

# Connect to your DagsHub repo
dagshub.init(repo_owner="shreychauhan02", repo_name="my-mlflow-project", mlflow=True)


Initialized MLflow to track repo "shreychauhan02/my-mlflow-project"

Repository shreychauhan02/my-mlflow-project initialized!

In [61]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

with mlflow.start_run():
    mlflow.set_tag('mlflow.runName','RandomForest_Baseline_TrainTestSplit')
    mlflow.set_tag('experiment_type','baseline')
    mlflow.set_tag('model_type','RandomForestClassifier')

    mlflow.set_tag('description','Baseline model RandomForest for sentiment analysis using Bag of Words with a leak-free train/test split (vectorizer fit on train only)')

    mlflow.log_param('vectorizer_type','CountVectorizer')
    mlflow.log_param('vectorizer_max_features',vectorizer.max_features)
    print("Test run logged successfully!")

    # Fit the vectorizer ONLY on the training text, then transform val/test with it
    X_train = vectorizer.fit_transform(X_train).toarray()
    X_test = vectorizer.transform(X_test).toarray()

    n_estimators=100
    max_depth=10

    mlflow.log_param('n_estimators',n_estimators)
    mlflow.log_param('max_depth',max_depth)

    model=RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth,random_state=42)
    model.fit(X_train,y_train)

    y_pred=model.predict(X_test)
    accuracy=accuracy_score(y_test,y_pred)
    mlflow.log_metric('accuracy',accuracy)

    classification_rep=classification_report(y_test,y_pred,output_dict=True)

    for label,metrics in classification_rep.items():
        if isinstance(metrics,dict):
            for metric ,value in metrics.items():
                mlflow.log_metric(f"{label}_{metric}",value)

    conf_matrix=confusion_matrix(y_test,y_pred)
    plt.figure(figsize=(10,6))
    sns.heatmap(conf_matrix,annot=True,fmt='d',cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")

    plt.savefig('confusion_matrix.png')
    mlflow.log_artifact('confusion_matrix.png')

    mlflow.sklearn.log_model(model,"random_forest_model")

    df.to_csv("dataset.csv",index=False)
    mlflow.log_artifact('dataset.csv')

print(f"accuracy_score:{accuracy}")

In [63]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

          -1       1.00      0.00      0.01      1650
           0       0.67      0.75      0.71      2555
           1       0.59      0.85      0.70      3154

    accuracy                           0.62      7359
   macro avg       0.76      0.53      0.47      7359
weighted avg       0.71      0.62      0.55      7359



In [64]:
df.to_csv("pre.csv",index=False)
